# 🔧 Save Encoders — FMCG Promotional Analytics
**One job: re-run preprocessing, save all encoders, upload to GCS.**  
No model retraining. Run all cells top to bottom (~5 mins).


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & CONFIG
# Only change needed: confirm the two filenames below match yours
# ══════════════════════════════════════════════════════════════════════
import os, glob, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# ── File paths (same folder as this notebook) ─────────────────────────
UK_FILE   = 'TPA UK - Cleaned - All - Digi Challenge.xlsx'
INDO_FILE = 'TPA - Sea - Cleaned - All - Digi Challenge.xlsx'

UK_SHEET   = 'Data Dump - Granular'
INDO_SHEET = 'Data Dump - Cleaned'

# ── GCS ───────────────────────────────────────────────────────────────
GCS_PROJECT = 'YOUR_GCP_PROJECT_ID'
GCS_BUCKET  = 'YOUR_BUCKET_NAME'

# ── Output folders ────────────────────────────────────────────────────
os.makedirs('models/encoders_uk',   exist_ok=True)
os.makedirs('models/encoders_sea', exist_ok=True)

print("✅ Config ready")
print(f"   UK   → {UK_FILE}  /  sheet: {UK_SHEET}")
print(f"   Sea → {INDO_FILE}  /  sheet: {INDO_SHEET}")


✅ Config ready
   UK   → TPA UK - Cleaned - All - Digi Challenge.xlsx  /  sheet: Data Dump - Granular
   Sea → TPA - Sea - Cleaned - All - Digi Challenge.xlsx  /  sheet: Data Dump - Cleaned


---
## 🇬🇧 Part 1 — UK Market

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CELL 2 — LOAD UK DATA
# ══════════════════════════════════════════════════════════════════════
print("Loading UK Excel file — this takes 1-2 minutes for a large file...")
df_UK = pd.read_excel(UK_FILE, sheet_name=UK_SHEET)
print(f"✅ Loaded: {df_UK.shape[0]:,} rows × {df_UK.shape[1]} cols")


Loading UK Excel file — this takes 1-2 minutes for a large file...
✅ Loaded: 279,007 rows × 45 cols


In [3]:
# ══════════════════════════════════════════════════════════════════════
# CELL 3 — UK CLEANING  (exact mirror of your original notebook)
# ══════════════════════════════════════════════════════════════════════
TARGET = 'ActualPromoSalesVolumeSellOut'

# Step 1: Remove zero sell-out rows → df_exec
df_exec = df_UK[df_UK[TARGET] > 0].copy()
print(f"After removing zero sell-out : {len(df_exec):,}")

# Step 2: Keep only Executed promos (drop InFlight etc.)
df_exec = df_exec[df_exec['PromotionStatus'] == 'Executed'].copy()
print(f"After keeping Executed only  : {len(df_exec):,}")

# Step 3: Fill missing PromoFeature
df_exec['PromoFeature'] = df_exec['PromoFeature'].fillna('Unknown')

# Step 4: Drop rows with no planned data
df_exec = df_exec.dropna(subset=['PlannedPromoSalesVolumeSellIn']).copy()
print(f"After dropping missing plan  : {len(df_exec):,}")

# Step 5: Build df_uk_valid — rows where planned volume > 0
df_uk_valid = df_UK[df_UK['PlannedPromoSalesVolumeSellIn'] > 0].copy()
df_uk_valid = df_uk_valid[df_uk_valid[TARGET] > 0].copy()
df_uk_valid = df_uk_valid[df_uk_valid['PromotionStatus'] == 'Executed'].copy()
df_uk_valid = df_uk_valid.dropna(subset=['PlannedPromoSalesVolumeSellIn']).copy()
df_uk_valid['PromoFeature'] = df_uk_valid['PromoFeature'].fillna('Unknown')

# Planning accuracy columns (kept for context, excluded from features)
df_uk_valid['PlanningAccuracyRatio'] = (
    df_uk_valid[TARGET] / df_uk_valid['PlannedPromoSalesVolumeSellIn']
)
cap_val = df_uk_valid['PlanningAccuracyRatio'].quantile(0.99)
df_uk_valid['PlanningAccuracyRatioCapped'] = df_uk_valid['PlanningAccuracyRatio'].clip(upper=cap_val)
df_uk_valid['PromoFailed']  = (df_uk_valid[TARGET] == 0).astype(int)
df_uk_valid['IsPipeFill']   = (df_uk_valid['PromoMechanic'] == 'Pipe Fill').astype(int)

print(f"\n✅ df_uk_valid shape: {df_uk_valid.shape}")


After removing zero sell-out : 226,019
After keeping Executed only  : 225,861
After dropping missing plan  : 218,820

✅ df_uk_valid shape: (189963, 49)


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 4 — UK FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════

# Derived numeric features
df_uk_valid['PlannedIncrementalVolume'] = (
    df_uk_valid['PlannedPromoSalesVolumeSellIn'] -
    df_uk_valid['PlannedBaselineVolume']
)

df_uk_valid['PlannedUpliftRate'] = (
    df_uk_valid['PlannedPromoSalesVolumeSellIn'] /
    df_uk_valid['PlannedBaselineVolume'].replace(0, pd.NA)
).clip(upper=10)

df_uk_valid['PlannedTTSTotal'] = (
    df_uk_valid['PlannedTTSOnSpend'] +
    df_uk_valid['PlannedTTSOffSpend'] +
    df_uk_valid['PlannedEventSpend']
).clip(lower=0)

df_uk_valid['PlannedCostPerUnit'] = (
    df_uk_valid['PlannedTTSTotal'] /
    df_uk_valid['PlannedPromoSalesVolumeSellIn'].replace(0, pd.NA)
).clip(lower=0)

df_uk_valid['PlannedROI_proxy'] = (
    df_uk_valid['PlannedNetPromoGrossProfitsSellIn'] /
    df_uk_valid['PlannedTTSTotal'].replace(0, pd.NA)
).clip(lower=-10, upper=50)

# Duration fix — data stored as YYYY.WW decimal e.g. 2023.36
start_week = (df_uk_valid['InStoreStartWeek'] % 1 * 100).round()
end_week   = (df_uk_valid['InStoreEndWeek']   % 1 * 100).round()
df_uk_valid['PromoDurationWeeks'] = (end_week - start_week).clip(lower=0, upper=26)

df_uk_valid['IsDefensivePromo'] = (
    df_uk_valid['PlannedIncrementalVolume'] < 0
).astype(int)

# Fix ratio dtype bug (pd.NA can silently make columns 'object')
df_uk_valid['PlannedUpliftRate'] = pd.to_numeric(
    df_uk_valid['PlannedUpliftRate'], errors='coerce'
).fillna(1.0)
df_uk_valid['PlannedROI_proxy'] = pd.to_numeric(
    df_uk_valid['PlannedROI_proxy'], errors='coerce'
).fillna(df_uk_valid['PlannedROI_proxy'].median())

# Drop ID columns
df_uk_valid = df_uk_valid.drop(columns=['PromoIDText','PromoFlag'], errors='ignore')

print(f"✅ Feature engineering done. Shape: {df_uk_valid.shape}")
print(f"Null count: {df_uk_valid.isnull().sum().sum()}")


✅ Feature engineering done. Shape: (189963, 54)
Null count: 0


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 5 — UK DATE EXTRACTION
# ══════════════════════════════════════════════════════════════════════
date_cols = df_uk_valid.select_dtypes(include='datetime64').columns.tolist()
print(f"Date columns found: {date_cols}")

for col in date_cols:
    df_uk_valid[col + '_Month'] = df_uk_valid[col].dt.month
    df_uk_valid[col + '_Week']  = df_uk_valid[col].dt.isocalendar().week.astype(int)

df_uk_valid = df_uk_valid.drop(columns=date_cols)
print(f"✅ Extracted month+week from {len(date_cols)} date columns")
print(f"Shape now: {df_uk_valid.shape}")


Date columns found: ['InstoreStartDate', 'InstoreEndDate', 'ShipmentStartDate', 'ShipmentEndDate']
✅ Extracted month+week from 4 date columns
Shape now: (189963, 58)


In [6]:
# ══════════════════════════════════════════════════════════════════════
# CELL 6 — UK ONE-HOT ENCODING  + SAVE OHE MAP
# ══════════════════════════════════════════════════════════════════════
ohe_cols_uk = ['Customer', 'PromotionStatus', 'DivisionName_VG',
               'PromoMechanic', 'PromoFeature', 'CategoryName_VG']

# Only encode columns that exist in this dataframe
ohe_cols_uk = [c for c in ohe_cols_uk if c in df_uk_valid.columns]
print(f"OHE columns: {ohe_cols_uk}")

df_uk_valid = pd.get_dummies(df_uk_valid, columns=ohe_cols_uk, dtype=int)
print(f"Shape after OHE: {df_uk_valid.shape}")

# Save which dummy columns each original column produced
ohe_map_uk = {
    orig: [c for c in df_uk_valid.columns if c.startswith(f'{orig}_')]
    for orig in ohe_cols_uk
}
with open('models/ohe_cols_uk.pkl', 'wb') as f:
    pickle.dump(ohe_map_uk, f)

print("\n✅ OHE map saved → models/ohe_cols_uk.pkl")
for orig, dummies in ohe_map_uk.items():
    print(f"   {orig:<25} → {len(dummies)} columns: {dummies[:3]}...")


OHE columns: ['Customer', 'PromotionStatus', 'DivisionName_VG', 'PromoMechanic', 'PromoFeature', 'CategoryName_VG']
Shape after OHE: (189963, 103)

✅ OHE map saved → models/ohe_cols_uk.pkl
   Customer                  → 7 columns: ['Customer_ASDA STORES LTD.', 'Customer_BOOTS UK LIMITED', 'Customer_SAINSBURYS SUPERMARKETS LTD']...
   PromotionStatus           → 1 columns: ['PromotionStatus_Executed']...
   DivisionName_VG           → 2 columns: ['DivisionName_VG_FOODS CATEGORY', 'DivisionName_VG_HPC CATEGORY']...
   PromoMechanic             → 8 columns: ['PromoMechanic_EDLP', 'PromoMechanic_Loyalty', 'PromoMechanic_Multi-Buy']...
   PromoFeature              → 17 columns: ['PromoFeature_Check out end', 'PromoFeature_Event', 'PromoFeature_Free Standing Unit']...
   CategoryName_VG           → 16 columns: ['CategoryName_VG_BEVERAGE', 'CategoryName_VG_DEODORANT & FRAGRANCE', 'CategoryName_VG_DRESSING']...


In [7]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7 — UK LABEL ENCODING  + SAVE EACH ENCODER
# THE FIX: create a fresh LabelEncoder() for every column
# ══════════════════════════════════════════════════════════════════════
le_cols_uk = ['Brand_VG', 'WeekSkID', 'PromoShopperMechanic',
              'SegmentName_VG', 'FormName_VG', 'EBFName_VG',
              'SPF', 'SPFVName_VG']

le_cols_uk = [c for c in le_cols_uk if c in df_uk_valid.columns]

encoders_uk = {}
for col in le_cols_uk:
    le = LabelEncoder()                      # ← fresh encoder each time (this was the bug)
    df_uk_valid[col] = le.fit_transform(df_uk_valid[col].astype(str))
    encoders_uk[col] = le
    with open(f'models/encoders_uk/{col}_le.pkl', 'wb') as f:
        pickle.dump(le, f)
    print(f"   ✅ {col:<30} {len(le.classes_):>6} classes  saved")

print(f"\n✅ {len(encoders_uk)} UK label encoders saved")


   ✅ Brand_VG                           56 classes  saved
   ✅ WeekSkID                          129 classes  saved
   ✅ PromoShopperMechanic               88 classes  saved
   ✅ SegmentName_VG                    105 classes  saved
   ✅ FormName_VG                        91 classes  saved
   ✅ EBFName_VG                        346 classes  saved
   ✅ SPF                               655 classes  saved
   ✅ SPFVName_VG                      1798 classes  saved

✅ 8 UK label encoders saved


In [8]:
# ══════════════════════════════════════════════════════════════════════
# CELL 8 — UK FINAL CHECK + SAVE FEATURE LIST
# ══════════════════════════════════════════════════════════════════════

# Check no object columns remain
remaining = df_uk_valid.select_dtypes(include='object').columns.tolist()
if remaining:
    print(f"⚠️  Still object columns — need encoding: {remaining}")
else:
    print("✅ No object columns — all encoded")

print(f"Null values: {df_uk_valid.isnull().sum().sum()}")

# Save ordered feature column list (excluding leakage / target)
leakage_uk = ['ActualPromoSalesVolumeSellOut', 'ActualBaselineVolume',
              'ActualBaselineValue', 'PromoFailed',
              'PlanningAccuracyRatio', 'PlanningAccuracyRatioCapped', 'LogTarget']
feature_cols_uk = [c for c in df_uk_valid.columns if c not in leakage_uk]

with open('models/feature_cols_uk.pkl', 'wb') as f:
    pickle.dump(feature_cols_uk, f)

print(f"✅ Feature list saved: {len(feature_cols_uk)} features → models/feature_cols_uk.pkl")


✅ No object columns — all encoded
Null values: 0
✅ Feature list saved: 97 features → models/feature_cols_uk.pkl


---
## 🇮🇩 Part 2 — Southeast Asia Market

In [9]:
# ══════════════════════════════════════════════════════════════════════
# CELL 9 — LOAD SOUTHEAST ASIA DATA
# ══════════════════════════════════════════════════════════════════════
print("Loading Southeast Asia Excel file...")
df_sea_raw = pd.read_excel(INDO_FILE, sheet_name=INDO_SHEET)
print(f"✅ Loaded: {df_sea_raw.shape[0]:,} rows × {df_sea_raw.shape[1]} cols")


Loading Southeast Asia Excel file...
✅ Loaded: 460,869 rows × 48 cols


In [10]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 — SOUTHEAST ASIA CLEANING
# ══════════════════════════════════════════════════════════════════════
TARGET_INDO = 'ActualNetPromoSalesVolumeSellOut'

df_sea_valid = df_sea_raw[df_sea_raw[TARGET_INDO] > 0].copy()
print(f"After removing zero sell-out : {len(df_sea_valid):,}")

if 'PromotionStatus' in df_sea_valid.columns:
    df_sea_valid = df_sea_valid[df_sea_valid['PromotionStatus'] != 'InFlight'].copy()
    print(f"After dropping InFlight      : {len(df_sea_valid):,}")

if 'PromoFeature' in df_sea_valid.columns:
    df_sea_valid['PromoFeature'] = df_sea_valid['PromoFeature'].fillna('Unknown')

df_sea_valid = df_sea_valid.dropna(
    subset=['PlannedPromoSalesVolumeSellIn']
).copy()
print(f"After dropping missing plan  : {len(df_sea_valid):,}")

df_sea_valid = df_sea_valid.drop(columns=['PromoIDText','PromoFlag'], errors='ignore')

for col in ['PlannedUpliftRate', 'PlannedROI_proxy']:
    if col in df_sea_valid.columns:
        df_sea_valid[col] = pd.to_numeric(
            df_sea_valid[col], errors='coerce'
        ).fillna(1.0)

print(f"\n✅ Southeast Asia clean shape: {df_sea_valid.shape}")


After removing zero sell-out : 404,803
After dropping InFlight      : 404,803
After dropping missing plan  : 404,803

✅ Southeast Asia clean shape: (404803, 46)


In [11]:
# ══════════════════════════════════════════════════════════════════════
# CELL 11 — SOUTHEAST ASIA FEATURE ENGINEERING + DATE EXTRACTION
# ══════════════════════════════════════════════════════════════════════

# Derived features (same as UK where columns exist)
for col_pair in [
    ('PlannedIncrementalVolume', lambda d: d['PlannedPromoSalesVolumeSellIn'] - d['PlannedBaselineVolume']),
    ('PlannedTTSTotal',          lambda d: (d.get('PlannedTTSOnSpend', 0) + d.get('PlannedTTSOffSpend', 0)).clip(lower=0)),
]:
    fname, func = col_pair
    try:
        df_sea_valid[fname] = func(df_sea_valid)
    except Exception as e:
        print(f"Skipping {fname}: {e}")

# Date extraction — use strftime for Sea (isocalendar can be slow on large datasets)
date_cols_id = df_sea_valid.select_dtypes(include='datetime64').columns.tolist()

# Also try converting object columns that look like dates
for col in df_sea_valid.select_dtypes(include='object').columns:
    if 'date' in col.lower() or 'Date' in col:
        try:
            df_sea_valid[col] = pd.to_datetime(df_sea_valid[col], errors='coerce')
        except:
            pass

date_cols_id = df_sea_valid.select_dtypes(include='datetime64').columns.tolist()
for col in date_cols_id:
    df_sea_valid[col + '_Month'] = df_sea_valid[col].dt.month.astype('int16')
    df_sea_valid[col + '_Week']  = df_sea_valid[col].dt.strftime('%W').astype('int16')
df_sea_valid = df_sea_valid.drop(columns=date_cols_id, errors='ignore')

print(f"Extracted month+week from {len(date_cols_id)} date columns")
print(f"✅ Southeast Asia shape: {df_sea_valid.shape}")


Extracted month+week from 4 date columns
✅ Southeast Asia shape: (404803, 52)


In [12]:
# ══════════════════════════════════════════════════════════════════════
# CELL 12 — SOUTHEAST ASIA OHE + SAVE MAP
# ══════════════════════════════════════════════════════════════════════
ohe_cols_sea = ['PromotionStatus', 'PromoMechanic', 'PromoGroup',
                 'fundtype', 'Level1Name_PPH']

# Only encode columns that exist AND have ≤20 unique values
ohe_cols_sea = [c for c in ohe_cols_sea
                 if c in df_sea_valid.columns
                 and df_sea_valid[c].nunique() <= 20]
print(f"OHE columns: {ohe_cols_sea}")

df_sea_valid = pd.get_dummies(df_sea_valid, columns=ohe_cols_sea, dtype=int)

ohe_map_sea = {
    orig: [c for c in df_sea_valid.columns if c.startswith(f'{orig}_')]
    for orig in ohe_cols_sea
}
with open('models/ohe_cols_sea.pkl', 'wb') as f:
    pickle.dump(ohe_map_sea, f)

print(f"✅ Sea OHE map saved. Shape: {df_sea_valid.shape}")


OHE columns: ['PromotionStatus', 'PromoMechanic', 'PromoGroup', 'fundtype', 'Level1Name_PPH']
✅ Sea OHE map saved. Shape: (404803, 76)


In [13]:
# ══════════════════════════════════════════════════════════════════════
# CELL 13 — SOUTHEAST ASIA LABEL ENCODING + SAVE
# ══════════════════════════════════════════════════════════════════════
le_cols_sea = ['Customer', 'Brand', 'Category', 'WeekSkID',
                'PromoShopperMechanic', 'Level4Name_PPH', 'Level6Name_PPH',
                'CPG', 'ProductNameSku_PPH', 'SPF']

le_cols_sea = [c for c in le_cols_sea if c in df_sea_valid.columns]

encoders_sea = {}
for col in le_cols_sea:
    le = LabelEncoder()
    df_sea_valid[col] = le.fit_transform(df_sea_valid[col].astype(str))
    encoders_sea[col] = le
    with open(f'models/encoders_sea/{col}_le.pkl', 'wb') as f:
        pickle.dump(le, f)
    print(f"   ✅ {col:<35} {len(le.classes_):>6} classes  saved")

# Save feature list
leakage_sea = [TARGET_INDO, 'ActualBaselineVolume', 'ActualBaselineValue',
                'PromoFailed', 'PlanningAccuracyRatio', 'PlanningAccuracyRatioCapped']
feature_cols_sea = [c for c in df_sea_valid.columns if c not in leakage_sea]
with open('models/feature_cols_sea.pkl', 'wb') as f:
    pickle.dump(feature_cols_sea, f)

print(f"\n✅ {len(encoders_sea)} Sea encoders saved")
print(f"✅ {len(feature_cols_sea)} Sea features saved")


   ✅ Customer                                35 classes  saved
   ✅ Brand                                   48 classes  saved
   ✅ Category                                20 classes  saved
   ✅ WeekSkID                               143 classes  saved
   ✅ PromoShopperMechanic                    16 classes  saved
   ✅ Level4Name_PPH                          82 classes  saved
   ✅ Level6Name_PPH                         582 classes  saved
   ✅ CPG                                    137 classes  saved
   ✅ ProductNameSku_PPH                    1074 classes  saved
   ✅ SPF                                   1074 classes  saved

✅ 10 Sea encoders saved
✅ 73 Sea features saved


---
## ☁️ Part 3 — Upload to GCS

In [14]:
# ══════════════════════════════════════════════════════════════════════
# CELL 14 — UPLOAD ALL .pkl FILES TO GCS
# ══════════════════════════════════════════════════════════════════════
from google.cloud import storage

client = storage.Client(project=GCS_PROJECT)
bucket = client.bucket(GCS_BUCKET)

to_upload = sorted(glob.glob('models/**/*.pkl', recursive=True))
print(f"Uploading {len(to_upload)} files to gs://{GCS_BUCKET}/...\n")

for local_path in to_upload:
    blob_name = local_path.replace('\\', '/')
    bucket.blob(blob_name).upload_from_filename(local_path)
    kb = os.path.getsize(local_path) / 1024
    print(f"  ↑ {blob_name:<55} ({kb:.1f} KB)")

print(f"\n✅ All {len(to_upload)} files uploaded!")


Uploading 26 files to gs://YOUR_BUCKET_NAME/...



Forbidden: 403 POST https://storage.googleapis.com/upload/storage/v1/b/unilever-promo-ml-data/o?uploadType=multipart: {
  "error": {
    "code": 403,
    "message": "weatherbit-api-access@weatherbit-api.iam.gserviceaccount.com does not have storage.objects.create access to the Google Cloud Storage object. Permission 'storage.objects.create' denied on resource (or it may not exist).",
    "errors": [
      {
        "message": "weatherbit-api-access@weatherbit-api.iam.gserviceaccount.com does not have storage.objects.create access to the Google Cloud Storage object. Permission 'storage.objects.create' denied on resource (or it may not exist).",
        "domain": "global",
        "reason": "forbidden"
      }
    ]
  }
}
: ('Request failed with status code', 403, 'Expected one of', <HTTPStatus.OK: 200>)

In [ ]:
import pickle, xgboost as xgb

for market, pkl_path, json_path in [
    ('UK',        'models/xgb_uk_tuned.pkl',   'models/xgb_uk_tuned.json'),
    ('Southeast Asia', 'models/xgb_sea_tuned.pkl', 'models/xgb_sea_tuned.json'),
]:
    model = pickle.load(open(pkl_path, 'rb'))
    
    # Extract the raw booster and save that directly
    booster = model.get_booster()
    booster.save_model(json_path)
    print(f"✅ {market} booster saved → {json_path}")

[20:54:29] WARNING: C:\buildkite-agent\builds\buildkite-wseaws-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-wseaws\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.

✅ UK booster saved → models/xgb_uk_tuned.json
[20:54:29] WARNING: C:\buildkite-agent\builds\buildkite-wseaws-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-wseaws\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io

In [ ]:
import xgboost as xgb
import numpy as np

booster_uk = xgb.Booster()
booster_uk.load_model('models/xgb_uk_tuned.json')

# Get the exact feature names the model was trained with
model_features = booster_uk.feature_names

# Take one test row
test_row = df_uk_valid.iloc[int(len(df_uk_valid) * 0.8)]

# Align columns exactly to what model expects
# Missing columns get 0, extra columns get dropped
row_dict = test_row.to_dict()
aligned = {col: row_dict.get(col, 0) for col in model_features}
X_aligned = pd.DataFrame([aligned], columns=model_features).astype(float)

# Predict
dmat = xgb.DMatrix(X_aligned, feature_names=model_features)
log_pred = booster_uk.predict(dmat)[0]
pred_vol = float(np.expm1(log_pred))

actual = test_row['ActualPromoSalesVolumeSellOut']
print(f"Actual   : {actual:>12,.0f} units")
print(f"Predicted: {pred_vol:>12,.0f} units")
if pred_vol > 0:
    print("✅ Encoders working!")

Actual   :        7,878 units
Predicted:           35 units
✅ Encoders working!


---
## 🧪 Part 4 — Smoke Test

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 16 — FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════
all_files = sorted(glob.glob('models/**/*.pkl', recursive=True))

print("=" * 62)
print("  ALL SAVED ARTEFACTS")
print("=" * 62)
for f in all_files:
    kb = os.path.getsize(f) / 1024
    print(f"  {f:<50}  {kb:>7.1f} KB")
print("=" * 62)
print(f"  Total: {len(all_files)} files")
print()
print("✅ Done! Next step: build the Streamlit app.")


In [ ]:
import xgboost as xgb
booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')
print(booster.feature_names)

['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek', 'PromoShopperMechanic', 'SegmentName_VG', 'FormName_VG', 'EBFName_VG', 'Brand_VG', 'SPF', 'SPFVName_VG', 'ListPrice', 'PlannedPromoSalesVolumeSellIn', 'PlannedNetPromoGSVSellIn', 'PlannedTTSOnSpend', 'PlannedNetPromoNIVSellIn', 'PlannedTTSOffSpend', 'PlannedNetPromoTOSellIn', 'PlannedNetPromoGrossProfitsSellIn', 'PlannedNetPromoCOGSSellIn', 'PlannedBaselineVolume', 'PlannedBaseGSVSellIn', 'PlannedBaseTTSOnSpend', 'PlannedBaseNIVSellIn', 'PlannedBaseTTSOffSpend', 'PlannedBaseTOSellIn', 'PlannedBaseGrossProfitsSellIn', 'PlannedBaseCOGSSellIn', 'PlannedEventSpend', 'IsPipeFill', 'PlannedIncrementalVolume', 'PlannedUpliftRate', 'PlannedTTSTotal', 'PlannedCostPerUnit', 'PlannedROI_proxy', 'PromoDurationWeeks', 'IsDefensivePromo', 'Customer_ASDA STORES LTD.', 'Customer_BOOTS UK LIMITED', 'Customer_SAINSBURYS SUPERMARKETS LTD', 'Customer_T J MORRIS LTD', 'Customer_TESCO STORES LTD', 'Customer_WAITROSE LTD', 'Customer_WM

In [ ]:
# ── Save median feature values as defaults for the Streamlit app ─────────────
import json, pickle
import numpy as np

# Build the same feature matrix X that was used for training
leakage_uk = ['ActualPromoSalesVolumeSellOut', 'ActualBaselineVolume',
              'ActualBaselineValue', 'PromoFailed',
              'PlanningAccuracyRatio', 'PlanningAccuracyRatioCapped', 'LogTarget']

feat_cols = pickle.load(open('models/feature_cols_uk.pkl', 'rb'))
feat_cols = [c for c in feat_cols if c in df_uk_valid.columns]

X = df_uk_valid[feat_cols]

# Save the median of every column as a JSON file
medians = X.median().to_dict()
medians = {k: float(v) for k, v in medians.items()}

with open('models/feature_medians_uk.json', 'w') as f:
    json.dump(medians, f)

print(f"✅ Saved medians for {len(medians)} features → models/feature_medians_uk.json")
print(f"\nSample values:")
for k in list(medians.keys())[:8]:
    print(f"  {k:<40} {medians[k]:.3f}")

✅ Saved medians for 97 features → models/feature_medians_uk.json

Sample values:
  Level8Code                               350300.000
  TUEAN                                    8712561481632.000
  WeekSkID                                 67.000
  InStoreStartWeek                         2022.410
  InStoreEndWeek                           2022.470
  PromoShopperMechanic                     36.000
  SegmentName_VG                           43.000
  FormName_VG                              32.000


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FIX — Recompute medians from FULL encoded training data
# (not just the test set eval_df which was only 20% of rows)
# ══════════════════════════════════════════════════════════════════════
import json, pickle
import numpy as np

MODEL_FEATURES = [
    'Level8Code','TUEAN','WeekSkID','InStoreStartWeek','InStoreEndWeek',
    'PromoShopperMechanic','SegmentName_VG','FormName_VG','EBFName_VG',
    'Brand_VG','SPF','SPFVName_VG','ListPrice',
    'PlannedPromoSalesVolumeSellIn','PlannedNetPromoGSVSellIn',
    'PlannedTTSOnSpend','PlannedNetPromoNIVSellIn','PlannedTTSOffSpend',
    'PlannedNetPromoTOSellIn','PlannedNetPromoGrossProfitsSellIn',
    'PlannedNetPromoCOGSSellIn','PlannedBaselineVolume','PlannedBaseGSVSellIn',
    'PlannedBaseTTSOnSpend','PlannedBaseNIVSellIn','PlannedBaseTTSOffSpend',
    'PlannedBaseTOSellIn','PlannedBaseGrossProfitsSellIn','PlannedBaseCOGSSellIn',
    'PlannedEventSpend','IsPipeFill','PlannedIncrementalVolume',
    'PlannedUpliftRate','PlannedTTSTotal','PlannedCostPerUnit',
    'PlannedROI_proxy','PromoDurationWeeks','IsDefensivePromo',
    'Customer_ASDA STORES LTD.','Customer_BOOTS UK LIMITED',
    'Customer_SAINSBURYS SUPERMARKETS LTD','Customer_T J MORRIS LTD',
    'Customer_TESCO STORES LTD','Customer_WAITROSE LTD',
    'Customer_WM MORRISON SUPERMARKETS LIMITED',
    'PromotionStatus_Executed','PromotionStatus_InFlight',
    'DivisionName_VG_FOODS CATEGORY','DivisionName_VG_HPC CATEGORY',
    'PromoMechanic_EDLP','PromoMechanic_Loyalty','PromoMechanic_Multi-Buy',
    'PromoMechanic_Other','PromoMechanic_Pipe Fill',
    'PromoMechanic_Shopper Marketing','PromoMechanic_Special Packs / Offer',
    'PromoMechanic_TPR',
    'PromoFeature_Check out end','PromoFeature_Event',
    'PromoFeature_Free Standing Unit','PromoFeature_Gondola End',
    'PromoFeature_Hot Spot','PromoFeature_In queue fixture',
    'PromoFeature_Ladder Rack','PromoFeature_Mid Gondola',
    'PromoFeature_None Specified','PromoFeature_Online',
    'PromoFeature_Pallet Drop','PromoFeature_Plinth','PromoFeature_Shelf',
    'PromoFeature_Shipper/OFD','PromoFeature_Side Stack',
    'PromoFeature_Store Entrance',
    'CategoryName_VG_BEVERAGE','CategoryName_VG_DEODORANT & FRAGRANCE',
    'CategoryName_VG_DRESSING','CategoryName_VG_FABRIC CLEANING',
    'CategoryName_VG_FABRIC ENHANCER','CategoryName_VG_HAIR CARE',
    'CategoryName_VG_HEALTHY SNACKING','CategoryName_VG_HOME & HYGIENE',
    'CategoryName_VG_ICE CREAM CATEGORY',
    'CategoryName_VG_NON CORPORATE PC CATEGORY','CategoryName_VG_ORAL CARE',
    'CategoryName_VG_OTH NUTRITION','CategoryName_VG_PLANT BASED MEAT',
    'CategoryName_VG_SCRATCH COOKING AID','CategoryName_VG_SKIN CARE',
    'CategoryName_VG_SKIN CLEANSING',
    'InstoreStartDate_Month','InstoreStartDate_Week',
    'InstoreEndDate_Month','InstoreEndDate_Week',
    'ShipmentStartDate_Month','ShipmentStartDate_Week',
    'ShipmentEndDate_Month','ShipmentEndDate_Week',
]

# df_uk_valid is still in memory from earlier cells — full 219K rows
# Drop leakage columns to get the exact feature matrix
leakage = ['ActualPromoSalesVolumeSellOut','ActualBaselineVolume',
           'ActualBaselineValue','PromoFailed',
           'PlanningAccuracyRatio','PlanningAccuracyRatioCapped','LogTarget']

available = [c for c in MODEL_FEATURES if c in df_uk_valid.columns]
missing   = [c for c in MODEL_FEATURES if c not in df_uk_valid.columns]

print(f"Features found in df_uk_valid : {len(available)}/97")
if missing:
    print(f"Missing (will use 0)          : {missing}")

# Compute medians from full dataset
medians = {}
for col in MODEL_FEATURES:
    if col in df_uk_valid.columns:
        medians[col] = float(df_uk_valid[col].median())
    else:
        medians[col] = 0.0

# Spot check — these should be non-zero and realistic
print(f"\nSpot checks:")
print(f"  WeekSkID median             : {medians['WeekSkID']:.0f}")
print(f"  PlannedBaselineVolume median: {medians['PlannedBaselineVolume']:,.0f}")
print(f"  ListPrice median            : {medians['ListPrice']:.2f}")
print(f"  InstoreStartDate_Month      : {medians['InstoreStartDate_Month']:.0f}")
print(f"  Customer_TESCO STORES LTD   : {medians['Customer_TESCO STORES LTD']:.3f}")

# Save
with open('models/feature_medians_uk.json', 'w') as f:
    json.dump(medians, f, indent=2)
print(f"\n✅ Saved → models/feature_medians_uk.json  ({len(medians)} features)")
print(f"   Computed from {len(df_uk_valid):,} rows  (full dataset)")

Features found in df_uk_valid : 96/97
Missing (will use 0)          : ['PromotionStatus_InFlight']

Spot checks:
  WeekSkID median             : 67
  PlannedBaselineVolume median: 1,243
  ListPrice median            : 2.75
  InstoreStartDate_Month      : 7
  Customer_TESCO STORES LTD   : 0.000

✅ Saved → models/feature_medians_uk.json  (97 features)
   Computed from 189,963 rows  (full dataset)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DIAGNOSTIC — what does the booster actually predict with median row?
# ══════════════════════════════════════════════════════════════════════
import xgboost as xgb
import json, numpy as np, pandas as pd

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')

medians = json.load(open('models/feature_medians_uk.json'))

MODEL_FEATURES = [
    'Level8Code','TUEAN','WeekSkID','InStoreStartWeek','InStoreEndWeek',
    'PromoShopperMechanic','SegmentName_VG','FormName_VG','EBFName_VG',
    'Brand_VG','SPF','SPFVName_VG','ListPrice',
    'PlannedPromoSalesVolumeSellIn','PlannedNetPromoGSVSellIn',
    'PlannedTTSOnSpend','PlannedNetPromoNIVSellIn','PlannedTTSOffSpend',
    'PlannedNetPromoTOSellIn','PlannedNetPromoGrossProfitsSellIn',
    'PlannedNetPromoCOGSSellIn','PlannedBaselineVolume','PlannedBaseGSVSellIn',
    'PlannedBaseTTSOnSpend','PlannedBaseNIVSellIn','PlannedBaseTTSOffSpend',
    'PlannedBaseTOSellIn','PlannedBaseGrossProfitsSellIn','PlannedBaseCOGSSellIn',
    'PlannedEventSpend','IsPipeFill','PlannedIncrementalVolume',
    'PlannedUpliftRate','PlannedTTSTotal','PlannedCostPerUnit',
    'PlannedROI_proxy','PromoDurationWeeks','IsDefensivePromo',
    'Customer_ASDA STORES LTD.','Customer_BOOTS UK LIMITED',
    'Customer_SAINSBURYS SUPERMARKETS LTD','Customer_T J MORRIS LTD',
    'Customer_TESCO STORES LTD','Customer_WAITROSE LTD',
    'Customer_WM MORRISON SUPERMARKETS LIMITED',
    'PromotionStatus_Executed','PromotionStatus_InFlight',
    'DivisionName_VG_FOODS CATEGORY','DivisionName_VG_HPC CATEGORY',
    'PromoMechanic_EDLP','PromoMechanic_Loyalty','PromoMechanic_Multi-Buy',
    'PromoMechanic_Other','PromoMechanic_Pipe Fill',
    'PromoMechanic_Shopper Marketing','PromoMechanic_Special Packs / Offer',
    'PromoMechanic_TPR',
    'PromoFeature_Check out end','PromoFeature_Event',
    'PromoFeature_Free Standing Unit','PromoFeature_Gondola End',
    'PromoFeature_Hot Spot','PromoFeature_In queue fixture',
    'PromoFeature_Ladder Rack','PromoFeature_Mid Gondola',
    'PromoFeature_None Specified','PromoFeature_Online',
    'PromoFeature_Pallet Drop','PromoFeature_Plinth','PromoFeature_Shelf',
    'PromoFeature_Shipper/OFD','PromoFeature_Side Stack',
    'PromoFeature_Store Entrance',
    'CategoryName_VG_BEVERAGE','CategoryName_VG_DEODORANT & FRAGRANCE',
    'CategoryName_VG_DRESSING','CategoryName_VG_FABRIC CLEANING',
    'CategoryName_VG_FABRIC ENHANCER','CategoryName_VG_HAIR CARE',
    'CategoryName_VG_HEALTHY SNACKING','CategoryName_VG_HOME & HYGIENE',
    'CategoryName_VG_ICE CREAM CATEGORY',
    'CategoryName_VG_NON CORPORATE PC CATEGORY','CategoryName_VG_ORAL CARE',
    'CategoryName_VG_OTH NUTRITION','CategoryName_VG_PLANT BASED MEAT',
    'CategoryName_VG_SCRATCH COOKING AID','CategoryName_VG_SKIN CARE',
    'CategoryName_VG_SKIN CLEANSING',
    'InstoreStartDate_Month','InstoreStartDate_Week',
    'InstoreEndDate_Month','InstoreEndDate_Week',
    'ShipmentStartDate_Month','ShipmentStartDate_Week',
    'ShipmentEndDate_Month','ShipmentEndDate_Week',
]

# Test 1: pure median row
row = {col: float(medians.get(col, 0.0)) for col in MODEL_FEATURES}
X = pd.DataFrame([row], columns=MODEL_FEATURES).astype(float)
dmat = xgb.DMatrix(X)
log_pred = booster.predict(dmat, validate_features=False)[0]
print(f"Test 1 — Pure median row:")
print(f"  Raw log prediction : {log_pred:.4f}")
print(f"  Predicted volume   : {np.expm1(log_pred):,.0f} units")

# Test 2: use a real row from the training data
real_row = df_uk_valid[MODEL_FEATURES].iloc[1000]
X2 = pd.DataFrame([real_row], columns=MODEL_FEATURES).astype(float)
dmat2 = xgb.DMatrix(X2)
log_pred2 = booster.predict(dmat2, validate_features=False)[0]
actual = df_uk_valid['ActualPromoSalesVolumeSellOut'].iloc[1000]
print(f"\nTest 2 — Real row from training data (row 1000):")
print(f"  Raw log prediction : {log_pred2:.4f}")
print(f"  Predicted volume   : {np.expm1(log_pred2):,.0f} units")
print(f"  Actual volume      : {actual:,.0f} units")

Test 1 — Pure median row:
  Raw log prediction : 2.0742
  Predicted volume   : 7 units


KeyError: "['PromotionStatus_InFlight'] not in index"

In [ ]:
# ── Save column scaling groups for proportional inference ────────────────────
import json

scaling_groups = {
    # These columns scale with planned promo volume
    "promo_volume_cols": [
        "PlannedNetPromoGSVSellIn",
        "PlannedNetPromoNIVSellIn",
        "PlannedNetPromoTOSellIn",
        "PlannedNetPromoGrossProfitsSellIn",
        "PlannedNetPromoCOGSSellIn",
    ],
    # These scale with baseline volume
    "baseline_volume_cols": [
        "PlannedBaseGSVSellIn",
        "PlannedBaseNIVSellIn",
        "PlannedBaseTOSellIn",
        "PlannedBaseGrossProfitsSellIn",
        "PlannedBaseCOGSSellIn",
    ],
    # These scale with spend
    "spend_cols": [
        "PlannedBaseTTSOnSpend",
        "PlannedBaseTTSOffSpend",
        "PlannedEventSpend",
    ],
    # Store median values for the anchor columns
    "median_planned_volume": float(df_uk_valid["PlannedPromoSalesVolumeSellIn"].median()),
    "median_baseline_volume": float(df_uk_valid["PlannedBaselineVolume"].median()),
    "median_tts_total": float(df_uk_valid["PlannedTTSTotal"].median()),
}

with open("models/scaling_groups_uk.json", "w") as f:
    json.dump(scaling_groups, f, indent=2)

print("✅ Saved → models/scaling_groups_uk.json")
print(f"   Median planned volume : {scaling_groups['median_planned_volume']:,.0f} units")
print(f"   Median baseline volume: {scaling_groups['median_baseline_volume']:,.0f} units")
print(f"   Median TTS total      : {scaling_groups['median_tts_total']:,.0f}")

✅ Saved → models/scaling_groups_uk.json
   Median planned volume : 2,164 units
   Median baseline volume: 1,243 units
   Median TTS total      : 3,723


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RE-EXTRACT BOOSTER CLEANLY FROM ORIGINAL PKL
# One direct extraction — no intermediate conversions
# ══════════════════════════════════════════════════════════════════════
import pickle, xgboost as xgb, numpy as np, pandas as pd, json

# Load original pkl
with open('models/xgb_uk_tuned.pkl', 'rb') as f:
    uk_model = pickle.load(f)

# Extract booster and save directly as JSON
booster = uk_model.get_booster()
booster.save_model('models/xgb_uk_tuned.json')
print(f"✅ Booster saved → models/xgb_uk_tuned.json")

# Immediately verify with a real row from training data
medians = json.load(open('models/feature_medians_uk.json'))
MODEL_FEATURES = list(medians.keys())

# Test 1: median row
row = {col: float(medians[col]) for col in MODEL_FEATURES}
X = pd.DataFrame([row], columns=MODEL_FEATURES).astype(float)
dmat = xgb.DMatrix(X)
log_pred = booster.predict(dmat, validate_features=False)[0]
pred_vol = np.expm1(log_pred)
print(f"\nMedian row prediction  : {pred_vol:,.0f} units  (log={log_pred:.3f})")

# Test 2: real row from training data
real_row = df_uk_valid[[c for c in MODEL_FEATURES if c in df_uk_valid.columns]].iloc[5000]
missing = {c: float(medians.get(c,0)) for c in MODEL_FEATURES if c not in df_uk_valid.columns}
real_dict = {**real_row.to_dict(), **missing}
X2 = pd.DataFrame([real_dict], columns=MODEL_FEATURES).astype(float)
dmat2 = xgb.DMatrix(X2)
log_pred2 = booster.predict(dmat2, validate_features=False)[0]
pred2 = np.expm1(log_pred2)
actual2 = df_uk_valid['ActualPromoSalesVolumeSellOut'].iloc[5000]
print(f"Real row (idx 5000)    : predicted {pred2:,.0f} vs actual {actual2:,.0f} units")
print(f"Error                  : {abs(pred2-actual2)/max(actual2,1)*100:.1f}%")

✅ Booster saved → models/xgb_uk_tuned.json

Median row prediction  : 1,491 units  (log=7.308)
Real row (idx 5000)    : predicted 427 vs actual 294 units
Error                  : 45.3%


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
import json

# Load exactly as app does
booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')
medians = json.load(open('models/feature_medians_uk.json'))
scaling = json.load(open('models/scaling_groups_uk.json'))

MODEL_FEATURES = list(medians.keys())

# Simulate a user selecting Tesco, TPR, January, 10000 planned, 5000 baseline, £50000 spend
row = {col: float(medians.get(col, 0.0)) for col in MODEL_FEATURES}

# Apply user inputs
pv, bv, sp, dur, mw, mn = 10000, 5000, 50000, 2, 2, 1

row['PlannedPromoSalesVolumeSellIn'] = pv
row['PlannedBaselineVolume']         = bv
row['PlannedIncrementalVolume']      = pv - bv
row['PlannedUpliftRate']             = pv / max(bv, 1)
row['PlannedTTSTotal']               = sp
row['PlannedTTSOnSpend']             = sp * 0.6
row['PlannedTTSOffSpend']            = sp * 0.4
row['PlannedCostPerUnit']            = sp / max(pv, 1)
row['PromoDurationWeeks']            = dur
row['InstoreStartDate_Month']        = mn
row['InstoreStartDate_Week']         = mw
row['InstoreEndDate_Month']          = mn
row['InstoreEndDate_Week']           = mw + dur

# Proportional scaling
pv_ratio = pv / scaling['median_planned_volume']
bv_ratio = bv / scaling['median_baseline_volume']
sp_ratio = sp / scaling['median_tts_total']

for col in scaling['promo_volume_cols']:
    if col in row:
        row[col] = float(medians.get(col, 0.0)) * pv_ratio

for col in scaling['baseline_volume_cols']:
    if col in row:
        row[col] = float(medians.get(col, 0.0)) * bv_ratio

# Zero one-hots then set Tesco + TPR
for col in MODEL_FEATURES:
    if any(col.startswith(p) for p in ['Customer_','PromotionStatus_',
       'DivisionName_VG_','PromoMechanic_','PromoFeature_','CategoryName_VG_']):
        row[col] = 0.0

row['PromotionStatus_Executed']     = 1.0
row['Customer_TESCO STORES LTD']    = 1.0
row['DivisionName_VG_HPC CATEGORY'] = 1.0
row['PromoMechanic_TPR']            = 1.0
row['PromoFeature_None Specified']  = 1.0
row['CategoryName_VG_SKIN CARE']    = 1.0

X = pd.DataFrame([row], columns=MODEL_FEATURES).astype(float)
dmat = xgb.DMatrix(X)
log_pred = booster.predict(dmat, validate_features=False)[0]
pred_vol = max(float(np.expm1(log_pred)), 0)

print(f"Log prediction : {log_pred:.4f}")
print(f"Predicted volume: {pred_vol:,.0f} units")
print(f"Planned volume  : {pv:,} units")
print(f"Baseline volume : {bv:,} units")

Log prediction : 7.5053
Predicted volume: 1,817 units
Planned volume  : 10,000 units
Baseline volume : 5,000 units


In [ ]:
import os
print(os.path.abspath('models/xgb_uk_tuned.json'))
print(f"Size: {os.path.getsize('models/xgb_uk_tuned.json'):,} bytes")

c:\Panos\Data Analytics Consultancy-Claros\PROMOTIONAL ANALYTICS-Sellout Forecasting Trade ROI Optimization\Original datasets\models\xgb_uk_tuned.json
Size: 10,557,610 bytes


In [ ]:
import pickle, xgboost as xgb, numpy as np, pandas as pd, json

# Re-extract booster cleanly from original pkl
with open('models/xgb_uk_tuned.pkl', 'rb') as f:
    uk_model = pickle.load(f)

booster = uk_model.get_booster()
booster.save_model('models/xgb_uk_tuned.json')
print(f"✅ Booster saved")

# Verify immediately — should give ~1,500-2,000 units for median row
medians = json.load(open('models/feature_medians_uk.json'))
MODEL_FEATURES = list(medians.keys())
row = {col: float(medians[col]) for col in MODEL_FEATURES}
X = pd.DataFrame([row], columns=MODEL_FEATURES).astype(float)
dmat = xgb.DMatrix(X)
log_pred = booster.predict(dmat, validate_features=False)[0]
pred_vol = np.expm1(log_pred)
print(f"Median row prediction: {pred_vol:,.0f} units  (log={log_pred:.3f})")

✅ Booster saved
Median row prediction: 1,491 units  (log=7.308)


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
import json

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')
medians  = json.load(open('models/feature_medians_uk.json'))
scaling  = json.load(open('models/scaling_groups_uk.json'))

# Use booster's own feature order
feature_names = booster.feature_names
print(f"booster.feature_names is None: {feature_names is None}")
print(f"First 5 feature names: {feature_names[:5] if feature_names else 'NONE'}")

# Test 1 — pure medians, no modifications
row = {col: float(medians.get(col, 0.0)) for col in feature_names}
X = pd.DataFrame([row], columns=feature_names).astype(float)
pred1 = np.expm1(booster.predict(xgb.DMatrix(X), validate_features=False)[0])
print(f"\nTest 1 — pure medians    : {pred1:,.0f} units")

# Test 2 — add user inputs only (no proportional scaling)
row2 = dict(row)
row2['PlannedPromoSalesVolumeSellIn'] = 10000
row2['PlannedBaselineVolume']         = 5000
row2['PlannedIncrementalVolume']      = 5000
row2['PlannedUpliftRate']             = 2.0
row2['PlannedTTSTotal']               = 50000
row2['PlannedTTSOnSpend']             = 30000
row2['PlannedTTSOffSpend']            = 20000
row2['PlannedCostPerUnit']            = 5.0
X2 = pd.DataFrame([row2], columns=feature_names).astype(float)
pred2 = np.expm1(booster.predict(xgb.DMatrix(X2), validate_features=False)[0])
print(f"Test 2 — user inputs only: {pred2:,.0f} units")

# Test 3 — add proportional scaling on top
pv_ratio = 10000 / scaling['median_planned_volume']
bv_ratio = 5000  / scaling['median_baseline_volume']
row3 = dict(row2)
for col in scaling['promo_volume_cols']:
    if col in row3: row3[col] = float(medians.get(col, 0.0)) * pv_ratio
for col in scaling['baseline_volume_cols']:
    if col in row3: row3[col] = float(medians.get(col, 0.0)) * bv_ratio
X3 = pd.DataFrame([row3], columns=feature_names).astype(float)
pred3 = np.expm1(booster.predict(xgb.DMatrix(X3), validate_features=False)[0])
print(f"Test 3 — + scaling       : {pred3:,.0f} units")

# Test 4 — add one-hot zeroing
row4 = dict(row3)
for col in feature_names:
    if any(col.startswith(p) for p in ['Customer_','PromotionStatus_','DivisionName_VG_',
                                        'PromoMechanic_','PromoFeature_','CategoryName_VG_']):
        row4[col] = 0.0
row4['PromotionStatus_Executed']      = 1.0
row4['Customer_TESCO STORES LTD']     = 1.0
row4['DivisionName_VG_HPC CATEGORY']  = 1.0
row4['PromoMechanic_TPR']             = 1.0
row4['PromoFeature_None Specified']   = 1.0
row4['CategoryName_VG_SKIN CARE']     = 1.0
X4 = pd.DataFrame([row4], columns=feature_names).astype(float)
pred4 = np.expm1(booster.predict(xgb.DMatrix(X4), validate_features=False)[0])
print(f"Test 4 — + one-hots      : {pred4:,.0f} units")

booster.feature_names is None: False
First 5 feature names: ['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek']

Test 1 — pure medians    : 1,491 units
Test 2 — user inputs only: 3,048 units
Test 3 — + scaling       : 4,731 units
Test 4 — + one-hots      : 4,621 units


: 

In [ ]:
import xgboost as xgb

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')  # load existing

# Re-save as portable JSON (works cross-platform Wseaws→Linux)
model_bytes = booster.save_raw(raw_format='json')
with open('models/xgb_uk_tuned.json', 'wb') as f:
    f.write(model_bytes)

print(f"Saved: {len(model_bytes):,} bytes")

# Verify it loads back correctly
b2 = xgb.Booster()
b2.load_model('models/xgb_uk_tuned.json')
print(f"Reloaded OK — feature count: {len(b2.feature_names)}")

Saved: 10,557,610 bytes
Reloaded OK — feature count: 97


In [ ]:
import xgboost as xgb
import numpy as np, pandas as pd, json

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')

# Save as UBJ — truly cross-platform binary format
booster.save_model('models/xgb_uk_tuned.ubj')
print(f"Saved UBJ: {__import__('os').path.getsize('models/xgb_uk_tuned.ubj'):,} bytes")

# Verify
b2 = xgb.Booster()
b2.load_model('models/xgb_uk_tuned.ubj')
medians = json.load(open('models/feature_medians_uk.json'))
row = {col: float(medians[col]) for col in b2.feature_names}
X = pd.DataFrame([row]).astype(float)
pred = np.expm1(b2.predict(xgb.DMatrix(X), validate_features=False)[0])
print(f"Verification: {pred:,.0f} units  ← must be ~1,491")

Saved UBJ: 6,800,975 bytes
Verification: 1,491 units  ← must be ~1,491


In [ ]:
import xgboost as xgb, json

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')

feature_names = booster.feature_names
print(f"Total features: {len(feature_names)}")
print(feature_names)

# Save to file
with open('models/feature_names_uk.json', 'w') as f:
    json.dump(feature_names, f)
print("✅ Saved → models/feature_names_uk.json")

Total features: 97
['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek', 'PromoShopperMechanic', 'SegmentName_VG', 'FormName_VG', 'EBFName_VG', 'Brand_VG', 'SPF', 'SPFVName_VG', 'ListPrice', 'PlannedPromoSalesVolumeSellIn', 'PlannedNetPromoGSVSellIn', 'PlannedTTSOnSpend', 'PlannedNetPromoNIVSellIn', 'PlannedTTSOffSpend', 'PlannedNetPromoTOSellIn', 'PlannedNetPromoGrossProfitsSellIn', 'PlannedNetPromoCOGSSellIn', 'PlannedBaselineVolume', 'PlannedBaseGSVSellIn', 'PlannedBaseTTSOnSpend', 'PlannedBaseNIVSellIn', 'PlannedBaseTTSOffSpend', 'PlannedBaseTOSellIn', 'PlannedBaseGrossProfitsSellIn', 'PlannedBaseCOGSSellIn', 'PlannedEventSpend', 'IsPipeFill', 'PlannedIncrementalVolume', 'PlannedUpliftRate', 'PlannedTTSTotal', 'PlannedCostPerUnit', 'PlannedROI_proxy', 'PromoDurationWeeks', 'IsDefensivePromo', 'Customer_ASDA STORES LTD.', 'Customer_BOOTS UK LIMITED', 'Customer_SAINSBURYS SUPERMARKETS LTD', 'Customer_T J MORRIS LTD', 'Customer_TESCO STORES LTD', 'Customer_WAITROSE

In [21]:
import subprocess, requests, json, numpy as np

# Get access token directly from gcloud
token = "ya29.a0Aa7MYirF4eCJq6-qVIoSGcyUOx_yYGNVixE9sfrVFDsZmDEUaxsAGG6zY5RJ1u4iPFif3JkilJJgJyoFjmaIJxgaIguM6ImaGf0yK5c9Hc5zXGTD7P6q3yQpMXV8YAjw7yQH8zcRzVkad1a34aoD-ddAf6vjg7AaQXg-KSIQjtapC_-j0GFIcpzj-55e5iL6dLyTHQ0QAJliIAaCgYKAScSARcSFQHGX2MiZWp-LGOZJAi2SBQMMw0Daw0213"

# Build the request
endpoint_id = "6178114055031488512"
url = f"https://YOUR_REGION-aiplatform.googleapis.com/v1/projects/128825737789/locations/YOUR_REGION/endpoints/{endpoint_id}:predict"

instance = [float(row[col]) for col in MODEL_FEATURES]

response = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"instances": [instance]}
)

print("Status code:", response.status_code)
result = response.json()
print("Full response:", json.dumps(result, indent=2)[:500])

raw = result["predictions"][0]
print(f"\nRaw prediction:  {raw}")
print(f"After expm1:     {np.expm1(raw):,.0f} units")

Status code: 200
Full response: {
  "predictions": [
    2.757719278335571
  ],
  "deployedModelId": "4934056230621544448",
  "model": "projects/128825737789/locations/YOUR_REGION/models/1906056183307829248",
  "modelDisplayName": "uk-promo-forecaster-v1",
  "modelVersionId": "1"
}

Raw prediction:  2.757719278335571
After expm1:     15 units


In [22]:
import json

with open("models/feature_medians_uk.json") as f:
    medians = json.load(f)

# Check the key financial features
key_cols = [
    "PlannedPromoSalesVolumeSellIn",
    "PlannedBaselineVolume",
    "PlannedTTSTotal",
    "PlannedTTSOnSpend",
    "PlannedTTSOffSpend",
    "PlannedIncrementalVolume",
    "PlannedNetPromoGSVSellIn",
    "PlannedCostPerUnit",
]
for col in key_cols:
    print(f"{col:45s} = {medians.get(col, 'MISSING')}")

# Also check scaling groups
with open("models/scaling_groups_uk.json") as f:
    scaling = json.load(f)
print("\nmedian_planned_volume  :", scaling.get("median_planned_volume"))
print("median_baseline_volume :", scaling.get("median_baseline_volume"))
print("promo_volume_cols      :", scaling.get("promo_volume_cols"))
print("baseline_volume_cols   :", scaling.get("baseline_volume_cols"))

PlannedPromoSalesVolumeSellIn                 = 2163.63636
PlannedBaselineVolume                         = 1242.66
PlannedTTSTotal                               = 3722.53038
PlannedTTSOnSpend                             = 1334.1972
PlannedTTSOffSpend                            = 886.22373
PlannedIncrementalVolume                      = 567.93896
PlannedNetPromoGSVSellIn                      = 5496.31205
PlannedCostPerUnit                            = 1.9377447993288937

median_planned_volume  : 2163.63636
median_baseline_volume : 1242.66
promo_volume_cols      : ['PlannedNetPromoGSVSellIn', 'PlannedNetPromoNIVSellIn', 'PlannedNetPromoTOSellIn', 'PlannedNetPromoGrossProfitsSellIn', 'PlannedNetPromoCOGSSellIn']
baseline_volume_cols   : ['PlannedBaseGSVSellIn', 'PlannedBaseNIVSellIn', 'PlannedBaseTOSellIn', 'PlannedBaseGrossProfitsSellIn', 'PlannedBaseCOGSSellIn']


In [23]:
import xgboost as xgb, json, numpy as np, pandas as pd

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')

print("booster.feature_names:", booster.feature_names[:5] if booster.feature_names else "NONE")
print("Total features:", len(booster.feature_names) if booster.feature_names else "NONE")

# Pure median row test
medians = json.load(open('models/feature_medians_uk.json'))
feature_names = booster.feature_names if booster.feature_names else list(medians.keys())

row = {col: float(medians.get(col, 0.0)) for col in feature_names}
X = pd.DataFrame([row], columns=feature_names).astype(float)
dmat = xgb.DMatrix(X, feature_names=feature_names)
log_pred = booster.predict(dmat)[0]
print(f"Median row prediction: {np.expm1(log_pred):,.0f} units")

booster.feature_names: ['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek']
Total features: 97
Median row prediction: 1,491 units


In [24]:
# Save medians reordered to match booster.feature_names exactly
import json

medians = json.load(open('models/feature_medians_uk.json'))
ordered_medians = {col: medians.get(col, 0.0) for col in booster.feature_names}

with open('models/feature_medians_uk.json', 'w') as f:
    json.dump(ordered_medians, f)

print("Saved. Keys now match booster.feature_names order.")
print("First 5:", list(ordered_medians.keys())[:5])

Saved. Keys now match booster.feature_names order.
First 5: ['Level8Code', 'TUEAN', 'WeekSkID', 'InStoreStartWeek', 'InStoreEndWeek']


In [25]:
import xgboost as xgb, pandas as pd, numpy as np, json

booster = xgb.Booster()
booster.load_model('models/xgb_uk_tuned.json')
medians = json.load(open('models/feature_medians_uk.json'))
scaling = json.load(open('models/scaling_groups_uk.json'))

MODEL_FEATURES = booster.feature_names  # use booster order

pv, bv, sp, dur, mw, mn = 10000, 5000, 50000, 2, 2, 1

row = {col: float(medians.get(col, 0.0)) for col in MODEL_FEATURES}

pv_ratio = pv / max(float(scaling["median_planned_volume"]), 1)
bv_ratio = bv / max(float(scaling["median_baseline_volume"]), 1)
for col in scaling["promo_volume_cols"]:
    if col in row: row[col] = float(medians.get(col, 0.0)) * pv_ratio
for col in scaling["baseline_volume_cols"]:
    if col in row: row[col] = float(medians.get(col, 0.0)) * bv_ratio

row["PlannedPromoSalesVolumeSellIn"] = pv
row["PlannedBaselineVolume"]         = bv
row["PlannedIncrementalVolume"]      = pv - bv
row["PlannedUpliftRate"]             = pv / max(bv, 1)
row["PlannedTTSTotal"]               = sp
row["PlannedTTSOnSpend"]             = sp * 0.6
row["PlannedTTSOffSpend"]            = sp * 0.4
row["PlannedCostPerUnit"]            = sp / max(pv, 1)
row["PromoDurationWeeks"]            = dur

ONE_HOT_PREFIXES = ['Customer_','PromotionStatus_','DivisionName_VG_',
                    'PromoMechanic_','PromoFeature_','CategoryName_VG_']
for col in MODEL_FEATURES:
    if any(col.startswith(p) for p in ONE_HOT_PREFIXES):
        row[col] = 0.0

row["PromotionStatus_Executed"]     = 1.0
row["Customer_TESCO STORES LTD"]    = 1.0
row["DivisionName_VG_HPC CATEGORY"] = 1.0
row["PromoMechanic_TPR"]            = 1.0
row["PromoFeature_None Specified"]  = 1.0
row["CategoryName_VG_SKIN CARE"]    = 1.0

# Print key values
print(f"PlannedROI_proxy   : {row.get('PlannedROI_proxy')}")
print(f"PlannedCostPerUnit : {row.get('PlannedCostPerUnit')}")
print(f"PlannedTTSTotal    : {row.get('PlannedTTSTotal')}")

X = pd.DataFrame([row], columns=MODEL_FEATURES).astype(float)
dmat = xgb.DMatrix(X, feature_names=MODEL_FEATURES)
log_pred = booster.predict(dmat)[0]
print(f"\nPrediction: {np.expm1(log_pred):,.0f} units")

PlannedROI_proxy   : 0.29582670245572956
PlannedCostPerUnit : 5.0
PlannedTTSTotal    : 50000

Prediction: 2,955 units


In [27]:
from google.cloud import aiplatform
aiplatform.init(project='YOUR_GCP_PROJECT_ID', location='YOUR_REGION')
for e in aiplatform.Endpoint.list():
    print(e.display_name, e.resource_name)

PermissionDenied: 403 Permission 'aiplatform.endpoints.list' denied on resource '//aiplatform.googleapis.com/projects/unilever-promo-ml/locations/europe-west2' (or it may not exist). [reason: "IAM_PERMISSION_DENIED"
domain: "aiplatform.googleapis.com"
metadata {
  key: "resource"
  value: "projects/unilever-promo-ml/locations/europe-west2"
}
metadata {
  key: "permission"
  value: "aiplatform.endpoints.list"
}
]